# 03 · RAG Engine and the MCP server

Two of the agent's dependencies, used directly: query the RAG corpora the way the agent does, then speak the
MCP protocol to the tool server with plain `curl`.

In [ ]:
from nbsetup import *

## RAG Engine: query a corpus

`retrieval_query` embeds the question, finds the nearest chunks and returns them with a relevance score. The
corpus names come from the `.env` file that setup step 5 wrote.

In [ ]:
from dotenv import dotenv_values
import vertexai
from vertexai import rag

env = dotenv_values(REPO / "growth-risk-agent/.env")
vertexai.init(project=PROJECT, location=REGION)

for corpus, question in [
    (env["POLICY_RAG_CORPUS"], "What approval is needed for a discount above 20%?"),
    (env["SUPPORT_RAG_CORPUS"], "What should we do when checkout drop-off exceeds 50%?"),
]:
    print("Q:", question)
    resp = rag.retrieval_query(
        rag_resources=[rag.RagResource(rag_corpus=corpus)],
        text=question,
        rag_retrieval_config=rag.RagRetrievalConfig(top_k=2),
    )
    for c in resp.contexts.contexts:
        print(f"  [{c.source_display_name}] score={c.score:.3f}: {c.text[:300]!r}")
    print()

How the agent calls the same API, as an explicit function tool:

In [ ]:
show("growth-risk-agent/app/tools/rag_tools.py", "def _search(")

**What to notice**

- An earlier version used ADK's built-in `VertexAiRagRetrieval` tool. With it, Gemini decides whether to retrieve at all, and the agent invented ticket IDs and policy sections; nothing in the trace showed it.
- As a function tool, every search shows up in the trace, and the instructions can say "quote only the returned chunks".

## MCP: a remote tool server

MCP (Model Context Protocol) standardises how agents discover and call tools. This server exposes one tool, the
system's only write action. It speaks MCP over streamable HTTP, so it is an ordinary network service.

In [ ]:
show("mcp-server/server.py", "@mcp.tool()")

### Talk to it locally with curl

Three JSON-RPC calls: `initialize` (the server returns a session ID), `tools/list` (what ADK's `McpToolset`
discovers) and `tools/call` (what happens after a human approves).

In [ ]:
%%bash
cd "$REPO/mcp-server"
pkill -f "python server.py" 2>/dev/null; sleep 1
PORT=8090 nohup uv run -q --with "mcp>=1.24,<2" python server.py > /tmp/mcp.log 2>&1 &
for i in $(seq 1 60); do curl -s -o /dev/null http://127.0.0.1:8090/mcp && break; sleep 1; done
H=(-H "Content-Type: application/json" -H "Accept: application/json, text/event-stream")
mcp_call() { curl -s "${H[@]}" ${SID:+-H "mcp-session-id: $SID"} http://127.0.0.1:8090/mcp -d "$1"; }

SID=$(curl -s -D - "${H[@]}" http://127.0.0.1:8090/mcp -o /dev/null \
  -d '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-06-18","capabilities":{},"clientInfo":{"name":"curl","version":"1"}}}' \
  | grep -i "^mcp-session-id" | awk '{print $2}' | tr -d '\r')
echo "session: $SID"
mcp_call '{"jsonrpc":"2.0","method":"notifications/initialized"}' >/dev/null
echo "--- tools/list"; mcp_call '{"jsonrpc":"2.0","id":2,"method":"tools/list"}' | sed -n 's/^data: //p' | python3 -m json.tool | head -30
echo "--- tools/call"; mcp_call '{"jsonrpc":"2.0","id":3,"method":"tools/call","params":{"name":"create_remediation_action_mcp","arguments":{"action":"escalate_checkout_fix","campaign_id":"CAMP-2025-042","reason":"notebook test"}}}' | sed -n 's/^data: //p' | python3 -m json.tool | head -20
pkill -f "python server.py"

### Call the deployed copy

The Cloud Run copy is private. Without a token it returns 403. With an **ID token** whose audience is the service
URL, minted by impersonating the caller service account (exactly what the agent does), it answers.

In [ ]:
%%bash
echo "without a token: HTTP $(curl -s -o /dev/null -w '%{http_code}' -X POST "$MCP_URL/mcp")"
TOKEN=$(gcloud auth print-identity-token --impersonate-service-account "$MCP_CALLER" --audiences "$MCP_URL" --include-email 2>/dev/null)
H=(-H "Authorization: Bearer $TOKEN" -H "Content-Type: application/json" -H "Accept: application/json, text/event-stream")
SID=$(curl -s -D - "${H[@]}" "$MCP_URL/mcp" -o /dev/null \
  -d '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-06-18","capabilities":{},"clientInfo":{"name":"curl","version":"1"}}}' \
  | grep -i "^mcp-session-id" | awk '{print $2}' | tr -d '\r')
curl -s "${H[@]}" -H "mcp-session-id: $SID" "$MCP_URL/mcp" -d '{"jsonrpc":"2.0","method":"notifications/initialized"}' >/dev/null
echo "with a token, tools/list:"; curl -s "${H[@]}" -H "mcp-session-id: $SID" "$MCP_URL/mcp" -d '{"jsonrpc":"2.0","id":2,"method":"tools/list"}' | grep -o '"name":"[^"]*"'

How the agent builds that authenticated client:

In [ ]:
show("growth-risk-agent/app/tools/gcp_auth.py", "def mcp_http_client_factory_for(")

Next: [04 · Deploy](04_deploy.ipynb).